# 3. StructuredOutputParser (+ ResponseSchema)

The **no-Pydantic, minimal-boilerplate** way to get a dict of named fields out of the model. You list
the fields you want as `ResponseSchema` objects; the parser handles instructions + parsing.

---

## 1. Simple Definition

> **Kid version:** Instead of building a fancy form class, you jot a quick **checklist** of the boxes
> you want — "I need an `answer` and a `source`." This parser turns that checklist into instructions
> for the AI and reads the reply back into a dictionary.

**Professional definition:** `StructuredOutputParser` builds format instructions and parses model
output into a `dict`, using a list of **`ResponseSchema`** items (each = one field with a `name`,
`description`, and optional `type`). No Pydantic model needed.

```python
from langchain.output_parsers import ResponseSchema, StructuredOutputParser

schemas = [
    ResponseSchema(name="answer", description="the answer to the question"),
    ResponseSchema(name="source", description="the source used, if any"),
]
parser = StructuredOutputParser.from_response_schemas(schemas)
```

---

## 2. Why Does It Exist?

**The problem:** You want a few named fields (not just a string or list), but pulling in Pydantic feels
heavy for a quick task, and beginners may not know Pydantic yet.

### Before (Pydantic for 2 fields feels like overkill)

```python
class QA(BaseModel):
    answer: str = Field(description="the answer")
    source: str = Field(description="the source")
parser = PydanticOutputParser(pydantic_object=QA)
```

### After (just list the fields)

```python
parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(name="answer", description="the answer"),
    ResponseSchema(name="source", description="the source"),
])
```

Less ceremony for simple, flat outputs. Trade-off: **no type validation** (values come back as
strings) and **no nesting** — for those, use `PydanticOutputParser`.

---

## 3. Real-Life Analogy

A **sticky-note checklist** 📝 instead of an official printed form. For "grab the *answer* and the
*source*", a quick two-item note is faster than designing a formal document. The notes are your
`ResponseSchema`s.

---

## 4. Where It Fits in LangChain Architecture

```
BaseOutputParser
    │
    ▼
StructuredOutputParser        ← built from a list of ResponseSchema → dict
        ▲
        │ made of
   ResponseSchema(name, description, type)   ← one per field
```

- `ResponseSchema` = description of **one field**.
- `StructuredOutputParser` = assembles fields into instructions + a dict parser.
- Simpler cousin of `PydanticOutputParser` (dict, no validation, no nesting).

---

## 5. Internal Working

```
  ResponseSchemas: [answer, source]
        │
        ▼
  get_format_instructions()  → a fenced JSON template:
        ```json
        {
          "answer": string  // the answer to the question
          "source": string  // the source used, if any
        }
        ```
        │  (injected into the prompt)
        ▼
  model replies with that JSON (inside ```json fences)
        │
        ▼
  parse(text)  → strip fences, json.loads → dict
        │
        ▼
  {"answer": "...", "source": "..."}
```

---

## 6. Attributes / Methods

### `ResponseSchema(name, description, type)`

**Definition:** Describes one output field: `name` (dict key), `description` (instruction to the
model), optional `type` (e.g. `"string"`, `"list"`; default `"string"`).

**Why it exists:** The minimal unit of "a field I want back."

**When developers use it:** One per desired field.

**Real-life use case:** One line on your sticky-note checklist.

```python
ResponseSchema(name="tags", description="topic tags", type="list")
```

---

### `StructuredOutputParser.from_response_schemas()`

**Definition:** Classmethod that builds the parser from a list of `ResponseSchema`s.

```python
parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(name="answer", description="the answer"),
    ResponseSchema(name="confidence", description="0-1 confidence"),
])
```

---

### `get_format_instructions()`

**Definition:** Returns the fenced-JSON template telling the model which keys to produce.

```python
print(parser.get_format_instructions())
```

---

### `parse()`

**Definition:** Converts the model's JSON text into a `dict` (values are strings — no type coercion).

```python
parser.parse('```json\n{"answer":"Paris","confidence":"0.9"}\n```')
# {'answer': 'Paris', 'confidence': '0.9'}   ← note: "0.9" stays a string
```

---

## Putting it together

```python
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(name="answer", description="the answer to the user's question"),
    ResponseSchema(name="source", description="the source used, if any"),
])

prompt = PromptTemplate(
    template="Answer the question.\n{format_instructions}\nQuestion: {question}\n",
    input_variables=["question"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | ChatOpenAI(model="gpt-4o-mini") | parser
print(chain.invoke({"question": "Capital of France?"}))
# {'answer': 'Paris', 'source': '...'}
```

---

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

schema = [ResponseSchema(name='fact_1', description='Fact 1 about the topic'),
          ResponseSchema(name='fact_2', description='Fact 2 about the topic'),
          ResponseSchema(name='fact_3', description='Fact 3 about the topic')]

parser = StructuredOutputParser.from_response_schemas(schema)

topic_template = PromptTemplate(
                                template='Give 3 fact about {topic} \n {format_instruction}',
                                input_variables=['topic'],
                                partial_variables={'format_instruction':parser.get_format_instructions()}
                               )

print(topic_template.format(topic='black hole'))

Give 3 fact about black hole 
 The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"fact_1": string  // Fact 1 about the topic
	"fact_2": string  // Fact 2 about the topic
	"fact_3": string  // Fact 3 about the topic
}
```


In [4]:
chain = topic_template | llm | parser

llm_result = chain.invoke({'topic':'black hole'})

llm_result

{'fact_1': 'Black holes have an event horizon, a boundary beyond which nothing, not even light, can escape their gravitational pull.',
 'fact_2': 'They form when massive stars collapse under their own gravity, creating a region of spacetime with extreme density.',
 'fact_3': 'Supermassive black holes, found at the centers of galaxies, can weigh millions to billions of times more than the Sun.'}

In [5]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

# Initialize the LLM
llm = ChatOllama(model="qwen3:8b", temperature=0.2)

# Define the output schema
schema = [ResponseSchema(name="summary", description="A concise 3-5 sentence explanation of the topic."),
          ResponseSchema(name="key_concepts",description=("A list of 4-6 important concepts related to the topic. Each concept should include a short explanation.")),
          ResponseSchema(name="advantages", description=("A list of at least 3 major advantages or benefits of the topic.")),
          ResponseSchema(name="disadvantages", description=("A list of at least 3 limitations, disadvantages, or risks associated with the topic.")),
          ResponseSchema(name="applications", description=("A list of 4-5 real-world applications of the topic.")),
          ResponseSchema(name="difficulty_level", description=("Classify the topic as Beginner, Intermediate, or Advanced, and explain why.")),
          ResponseSchema(name="confidence_score", description=("A confidence score from 0 to 100 representing how confident you are in the overall answer.")),
          ResponseSchema(name="recommendation", description=("Explain whether someone should learn or explore this topic and why.")),
          ResponseSchema(name="follow_up_questions", description=("Provide 3 useful questions someone could ask next to learn more about the topic."))]

# Create the structured output parser
parser = StructuredOutputParser.from_response_schemas(schema)

# Create the prompt template
topic_template = PromptTemplate(
    template="""
You are an expert technical educator and research assistant.

Analyze the following topic:

Topic: {topic}

Target audience:
{audience}

Your job is to provide a structured and educational analysis.

Requirements:

1. Explain the topic clearly.
2. Identify the most important concepts.
3. Explain both benefits and limitations.
4. Give practical real-world applications.
5. Estimate the difficulty level.
6. Give a confidence score between 0 and 100.
7. Recommend whether the target audience should learn this topic.
8. Suggest useful follow-up questions.

Do not make unsupported claims.
Keep explanations technically accurate but easy to understand.

{format_instruction}
""",

    input_variables=["topic", "audience"],
    partial_variables={"format_instruction": parser.get_format_instructions()}
)

# Display the generated prompt
formatted_prompt = topic_template.format(topic="Large Language Models", audience="A Python developer who is new to Generative AI")
print(formatted_prompt)


You are an expert technical educator and research assistant.

Analyze the following topic:

Topic: Large Language Models

Target audience:
A Python developer who is new to Generative AI

Your job is to provide a structured and educational analysis.

Requirements:

1. Explain the topic clearly.
2. Identify the most important concepts.
3. Explain both benefits and limitations.
4. Give practical real-world applications.
5. Estimate the difficulty level.
6. Give a confidence score between 0 and 100.
7. Recommend whether the target audience should learn this topic.
8. Suggest useful follow-up questions.

Do not make unsupported claims.
Keep explanations technically accurate but easy to understand.

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"summary": string  // A concise 3-5 sentence explanation of the topic.
	"key_concepts": string  // A list of 4-6 important concepts related to the t

In [6]:
chain = topic_template | llm | parser

result = chain.invoke({"topic": "Large Language Models", "audience": "A Python developer who is new to Generative AI"})
print(result)

{'summary': 'Large Language Models (LLMs) are advanced AI systems trained on vast text data to generate human-like text, answer questions, and perform tasks like coding. They leverage deep learning architectures to understand and produce natural language, making them versatile tools for developers. Python developers can integrate LLMs into applications for automation, analysis, and creativity.', 'key_concepts': ['Transformer Architecture: The foundation of LLMs, using self-attention mechanisms to process text efficiently.', 'Training Data: Vast corpora of text (books, websites, etc.) used to teach models language patterns and knowledge.', 'Parameters: Internal variables (millions to trillions) that capture language understanding and generation capabilities.', 'Fine-tuning: Adapting pre-trained LLMs to specific tasks (e.g., code generation) with task-specific data.', 'Inference: Using a trained model to generate outputs (e.g., text, code) based on input prompts.'], 'advantages': ['Multi